# Phase M5 — Video B v3: Improved DDPM
## v2 fixes + EMA + Elastic Aug + Self-Attention + DDIM + Anatomical Validation
---

In [ ]:
import os,time,math,gc,warnings,copy
import numpy as np
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torch.cuda.amp import GradScaler,autocast
from pathlib import Path
from scipy.ndimage import zoom,gaussian_filter,map_coordinates,label as scipy_label,binary_erosion
warnings.filterwarnings('ignore')
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CFG={'res':64,'n_cls':5,'bs':8,'T':50,'lr':1e-4,'epochs':500,
     'patience':50,'val_split':0.15,'base_ch':96,'warmup':10,'ema_decay':0.999}
OUT=Path('/kaggle/working/video_train_v3');OUT.mkdir(parents=True,exist_ok=True)
print(f'Device:{device} Config:{CFG}')

In [ ]:
# Load + downsample
tp=None
for p in Path('/kaggle/input').rglob('mu_glioma_triplets.npz'):tp=p;break
if tp is None:raise RuntimeError('not found')
d=np.load(tp);N=len(d['seg_start'])
def dseg(a):return zoom(a,(1,0.5,0.5),order=0).astype(np.uint8)
def dt1c(a):return zoom(a,(1,0.5,0.5),order=1).astype(np.float32)
seg_s=dseg(d['seg_start']);seg_e=dseg(d['seg_end']);seg_g=dseg(d['seg_gt'])
t1c_s=dt1c(d['t1c_start']);t_interp=d['t_interp'];types=d['types']
print(f'{N} samples downsampled to {seg_s.shape[1]}x{seg_s.shape[2]}')

In [ ]:
# Dataset with elastic augmentation + soft labels
def seg_to_soft(seg,C=5):
    oh=np.zeros((C,seg.shape[0],seg.shape[1]),dtype=np.float32)
    for c in range(C):oh[c]=(seg==c).astype(np.float32)
    return oh*1.9-0.95

def elastic_deform(arrays_2d,seg_arrays,sigma=5,alpha=3):
    shape=arrays_2d[0].shape
    dy=gaussian_filter(np.random.randn(*shape),sigma)*alpha
    dx=gaussian_filter(np.random.randn(*shape),sigma)*alpha
    y,x=np.meshgrid(np.arange(shape[0]),np.arange(shape[1]),indexing='ij')
    coords=[y+dy,x+dx]
    out_2d=[map_coordinates(a,coords,order=1,mode='nearest') for a in arrays_2d]
    out_seg=[map_coordinates(a,coords,order=0,mode='nearest') for a in seg_arrays]
    return out_2d,out_seg

class DS(Dataset):
    def __init__(s,ss,se,sg,ti,t1c,aug=True):
        s.ss,s.se,s.sg,s.ti,s.t1c,s.aug=ss,se,sg,ti,t1c,aug
    def __len__(s):return len(s.ss)
    def __getitem__(s,i):
        ss_r,se_r,sg_r=s.ss[i].copy(),s.se[i].copy(),s.sg[i].copy()
        t1c=s.t1c[i].copy();t=np.float32(s.ti[i])
        if s.aug:
            if np.random.rand()>0.5:
                [t1c],[ss_r,se_r,sg_r]=elastic_deform([t1c],[ss_r,se_r,sg_r])
            if np.random.rand()>0.5:
                ss_r=ss_r[::-1].copy();se_r=se_r[::-1].copy()
                sg_r=sg_r[::-1].copy();t1c=t1c[::-1].copy()
            if np.random.rand()>0.5:
                ss_r=ss_r[:,::-1].copy();se_r=se_r[:,::-1].copy()
                sg_r=sg_r[:,::-1].copy();t1c=t1c[:,::-1].copy()
            if np.random.rand()>0.5:
                k=np.random.choice([1,2,3])
                ss_r=np.rot90(ss_r,k).copy();se_r=np.rot90(se_r,k).copy()
                sg_r=np.rot90(sg_r,k).copy();t1c=np.rot90(t1c,k).copy()
        ss_s=seg_to_soft(ss_r);se_s=seg_to_soft(se_r);sg_s=seg_to_soft(sg_r)
        return {'cond':torch.from_numpy(np.concatenate([ss_s,se_s],0)),
                'x0':torch.from_numpy(sg_s),'gt_labels':torch.from_numpy(sg_r.astype(np.int64)),
                't_interp':torch.tensor([t]),'t1c':torch.from_numpy(t1c)}

np.random.seed(42);idx=np.random.permutation(N);nv=int(N*CFG['val_split'])
tr=DS(seg_s[idx[nv:]],seg_e[idx[nv:]],seg_g[idx[nv:]],t_interp[idx[nv:]],t1c_s[idx[nv:]],True)
va=DS(seg_s[idx[:nv]],seg_e[idx[:nv]],seg_g[idx[:nv]],t_interp[idx[:nv]],t1c_s[idx[:nv]],False)
tl=DataLoader(tr,batch_size=CFG['bs'],shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
vl=DataLoader(va,batch_size=CFG['bs'],shuffle=False,num_workers=2,pin_memory=True)
print(f'Train:{len(tr)} Val:{len(va)}')

In [ ]:
# Cosine schedule T=50
def cosine_betas(T,s=0.008):
    t=torch.linspace(0,T,T+1)
    ac=torch.cos(((t/T)+s)/(1+s)*math.pi*0.5)**2
    ac=ac/ac[0];b=1-(ac[1:]/ac[:-1]);return torch.clip(b,0.0001,0.9999)
T=CFG['T'];betas=cosine_betas(T).to(device)
alphas=1-betas;ac=torch.cumprod(alphas,0)
sac=torch.sqrt(ac);s1mac=torch.sqrt(1-ac)
acp=F.pad(ac[:-1],(1,0),value=1.0);pv=betas*(1-acp)/(1-ac)
def q_sample(x0,t,noise=None):
    if noise is None:noise=torch.randn_like(x0)
    return sac[t][:,None,None,None]*x0+s1mac[t][:,None,None,None]*noise,noise
print(f'Schedule: cosine T={T}')

In [ ]:
# DDPMv3: FiLM + Self-Attention at bottleneck
class SinPE(nn.Module):
    def __init__(s,d):super().__init__();s.d=d
    def forward(s,t):
        h=s.d//2;e=math.log(10000)/(h-1)
        e=torch.exp(torch.arange(h,device=t.device)*-e)
        e=t[:,None].float()*e[None,:];return torch.cat([e.sin(),e.cos()],-1)

class SelfAttn(nn.Module):
    def __init__(s,ch):
        super().__init__();s.norm=nn.GroupNorm(8,ch)
        s.qkv=nn.Conv2d(ch,ch*3,1);s.proj=nn.Conv2d(ch,ch,1)
    def forward(s,x):
        B,C,H,W=x.shape;h=s.norm(x)
        qkv=s.qkv(h).reshape(B,3,C,H*W);q,k,v=qkv[:,0],qkv[:,1],qkv[:,2]
        a=(q.transpose(-1,-2)@k)/(C**0.5);a=a.softmax(-1)
        h=(v@a.transpose(-1,-2)).reshape(B,C,H,W);return x+s.proj(h)

class FiLMRes(nn.Module):
    def __init__(s,ic,oc,cc,td):
        super().__init__()
        g1=min(8,ic) if ic%8==0 else(4 if ic%4==0 else 1)
        g2=min(8,oc) if oc%8==0 else(4 if oc%4==0 else 1)
        s.c1=nn.Sequential(nn.GroupNorm(g1,ic),nn.SiLU(),nn.Conv2d(ic,oc,3,1,1))
        s.c2=nn.Sequential(nn.GroupNorm(g2,oc),nn.SiLU(),nn.Conv2d(oc,oc,3,1,1))
        s.tp=nn.Sequential(nn.SiLU(),nn.Linear(td,oc))
        s.film=nn.Conv2d(cc,oc*2,1)
        s.sk=nn.Conv2d(ic,oc,1) if ic!=oc else nn.Identity()
    def forward(s,x,te,cf):
        h=s.c1(x);h=h+s.tp(te)[:,:,None,None]
        fs=s.film(cf);sc,sh=fs.chunk(2,dim=1);h=h*(1+sc)+sh
        return s.c2(h)+s.sk(x)

class CondEnc(nn.Module):
    def __init__(s,ic=10,B=96):
        super().__init__()
        s.e1=nn.Sequential(nn.Conv2d(ic,B,3,1,1),nn.SiLU())
        s.e2=nn.Sequential(nn.Conv2d(B,B*2,3,2,1),nn.SiLU())
        s.e3=nn.Sequential(nn.Conv2d(B*2,B*4,3,2,1),nn.SiLU())
        s.e4=nn.Sequential(nn.Conv2d(B*4,B*8,3,2,1),nn.SiLU())
    def forward(s,x):return[s.e1(x),s.e2(s.e1(x)),s.e3(s.e2(s.e1(x))),s.e4(s.e3(s.e2(s.e1(x))))]

class DDPMv3(nn.Module):
    def __init__(s,xc=5,cc=10,B=96,td=256):
        super().__init__()
        s.te=nn.Sequential(SinPE(td),nn.Linear(td,td),nn.SiLU())
        s.tie=nn.Sequential(nn.Linear(1,td),nn.SiLU(),nn.Linear(td,td))
        s.ce=CondEnc(cc,B)
        s.inp=nn.Sequential(nn.Conv2d(xc,B,3,1,1),nn.SiLU())
        s.e1=FiLMRes(B,B,B,td);s.e2=FiLMRes(B,B*2,B*2,td)
        s.e3=FiLMRes(B*2,B*4,B*4,td);s.e4=FiLMRes(B*4,B*8,B*8,td)
        s.attn=SelfAttn(B*8)  # NEW: attention at bottleneck
        s.dn=nn.MaxPool2d(2)
        s.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
        s.d3=FiLMRes(B*8+B*4,B*4,B*4,td)
        s.d2=FiLMRes(B*4+B*2,B*2,B*2,td)
        s.d1=FiLMRes(B*2+B,B,B,td)
        s.out=nn.Conv2d(B,xc,1)
    def forward(s,xn,td_,cond,ti):
        te=s.te(td_)+s.tie(ti);cf=s.ce(cond)
        x=s.inp(xn)
        e1=s.e1(x,te,cf[0]);e2=s.e2(s.dn(e1),te,cf[1])
        e3=s.e3(s.dn(e2),te,cf[2]);e4=s.e4(s.dn(e3),te,cf[3])
        e4=s.attn(e4)  # self-attention
        d3=s.d3(torch.cat([s.up(e4),e3],1),te,cf[2])
        d2=s.d2(torch.cat([s.up(d3),e2],1),te,cf[1])
        d1=s.d1(torch.cat([s.up(d2),e1],1),te,cf[0])
        return s.out(d1)

model=DDPMv3(B=CFG['base_ch']).to(device)
npar=sum(p.numel() for p in model.parameters())/1e6
print(f'DDPMv3: {npar:.1f}M params')
with torch.no_grad():
    o=model(torch.randn(2,5,64,64).to(device),torch.randint(0,T,(2,)).to(device),
            torch.randn(2,10,64,64).to(device),torch.rand(2,1).to(device))
    print(f'Shape:{o.shape} ok');del o;torch.cuda.empty_cache()

In [ ]:
# EMA + Hybrid Loss + Training
class EMA:
    def __init__(s,model,decay=0.999):
        s.shadow={k:v.clone() for k,v in model.state_dict().items()};s.decay=decay
    def update(s,model):
        for k,v in model.state_dict().items():s.shadow[k]=s.decay*s.shadow[k]+(1-s.decay)*v
    def apply(s,model):model.load_state_dict(s.shadow)
    def state_dict(s):return s.shadow
    def load_state_dict(s,sd):s.shadow=sd

CW=torch.tensor([0.1,2.0,2.0,0.1,3.0],device=device)
def predict_x0(xt,t,np_):return(xt-s1mac[t][:,None,None,None]*np_)/sac[t][:,None,None,None].clamp(min=1e-5)
def soft_dice(logits,target,smooth=1.0):
    p=torch.softmax(logits,1);toh=F.one_hot(target,5).permute(0,3,1,2).float()
    inter=(p*toh).sum((2,3));union=p.sum((2,3))+toh.sum((2,3))
    return 1-(2*inter+smooth)/(union+smooth)
def hybrid_loss(np_,noise,xt,t,gl,x0):
    tm=(gl>0).float().unsqueeze(1).expand_as(noise);w=1.0+49.0*tm
    mse=(w*(np_-noise)**2).mean()
    x0p=predict_x0(xt,t,np_)
    dl=soft_dice(x0p,gl)[:,1:].mean()
    ce=F.cross_entropy(x0p,gl,weight=CW)
    return mse+0.5*dl+0.3*ce
def dice_sc(p,g):
    r={}
    for n,pm,gm in[('WT',p>0,g>0),('TC',(p==1)|(p==4),(g==1)|(g==4)),('ET',p==4,g==4)]:
        r[n]=(2*(pm&gm).sum().float()/(pm.sum().float()+gm.sum().float()+1e-8)).item()
    return r

ema=EMA(model,CFG['ema_decay'])
opt=torch.optim.AdamW(model.parameters(),lr=CFG['lr'],weight_decay=1e-4)
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CFG['epochs'],eta_min=1e-6)
scaler=GradScaler()
best_dice=0.0;pat=0;tl_,vl_,vd_=[],[],[]
CKPT=OUT/'ddpm_v3_best.pth'
se=0
for p in Path('/kaggle/input').rglob('ddpm_v3_best.pth'):
    ck=torch.load(p,map_location=device,weights_only=False)
    model.load_state_dict(ck['model']);se=ck.get('epoch',0)+1
    best_dice=ck.get('best_dice',0)
    if 'ema' in ck:ema.load_state_dict(ck['ema'])
    print(f'Resumed ep{se} dice={best_dice:.4f}');break
print(f'Training {se}->{CFG["epochs"]}')
t0=time.time()
for ep in range(se,CFG['epochs']):
    # Warmup LR
    if ep<CFG['warmup']:
        for pg in opt.param_groups:pg['lr']=CFG['lr']*(ep+1)/CFG['warmup']
    model.train();el=[]
    for b in tl:
        cond=b['cond'].to(device);x0=b['x0'].to(device)
        gl=b['gt_labels'].to(device);ti=b['t_interp'].to(device)
        B_=cond.shape[0];t=torch.randint(0,T,(B_,),device=device)
        with autocast():
            xt,noise=q_sample(x0,t);np_=model(xt,t,cond,ti)
            loss=hybrid_loss(np_,noise,xt,t,gl,x0)
        opt.zero_grad();scaler.scale(loss).backward()
        scaler.unscale_(opt);nn.utils.clip_grad_norm_(model.parameters(),1.0)
        scaler.step(opt);scaler.update();ema.update(model);el.append(loss.item())
    if ep>=CFG['warmup']:sch.step()
    tl_.append(np.mean(el))
    # Val with EMA
    orig_sd=copy.deepcopy(model.state_dict());ema.apply(model);model.eval()
    vll=[];vd={'WT':[],'TC':[],'ET':[]}
    with torch.no_grad():
        for b in vl:
            cond=b['cond'].to(device);x0=b['x0'].to(device)
            gl=b['gt_labels'].to(device);ti=b['t_interp'].to(device)
            B_=cond.shape[0];t=torch.randint(0,T,(B_,),device=device)
            with autocast():
                xt,noise=q_sample(x0,t);np_=model(xt,t,cond,ti)
                vll.append(hybrid_loss(np_,noise,xt,t,gl,x0).item())
            x0p=predict_x0(xt,t,np_);pl=x0p.argmax(1)
            for i in range(B_):
                ds=dice_sc(pl[i],gl[i])
                for k in ds:vd[k].append(ds[k])
    model.load_state_dict(orig_sd)
    vl_.append(np.mean(vll));md_=np.mean([np.mean(vd[k]) for k in vd]);vd_.append(md_)
    imp=md_>best_dice
    if imp:
        best_dice=md_;pat=0
        torch.save({'model':model.state_dict(),'ema':ema.state_dict(),
                     'epoch':ep,'best_dice':best_dice,'cfg':CFG},str(CKPT))
    else:pat+=1
    if(ep+1)%10==0 or imp:
        print(f'  Ep{ep+1:3d}/{CFG["epochs"]} l={tl_[-1]:.4f}/{vl_[-1]:.4f} '
              f'WT={np.mean(vd["WT"]):.3f} TC={np.mean(vd["TC"]):.3f} ET={np.mean(vd["ET"]):.3f} '
              f'avg={md_:.3f} best={best_dice:.3f} pat={pat} {time.time()-t0:.0f}s')
    if pat>=CFG['patience']:print(f'  Early stop ep{ep+1}');break
print(f'Done. Best:{best_dice:.4f}')

In [ ]:
# Curves
import matplotlib;matplotlib.use('Agg');import matplotlib.pyplot as plt
fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
a1.plot(tl_,label='Train',color='#1565C0');a1.plot(vl_,label='Val',color='#E53935')
a1.set_xlabel('Epoch');a1.set_ylabel('Loss');a1.legend();a1.grid(True,alpha=0.3);a1.set_title('Loss')
a2.plot(vd_,label='Avg Dice',color='#2E7D32',lw=2)
a2.axhline(best_dice,ls='--',color='gray',label=f'Best:{best_dice:.3f}')
a2.set_xlabel('Epoch');a2.set_ylabel('Dice');a2.legend();a2.grid(True,alpha=0.3);a2.set_title('Dice')
plt.tight_layout();plt.savefig(str(OUT/'curves_v3.png'),dpi=150);plt.show()

In [ ]:
# DDIM sampling + load EMA weights
ck=torch.load(str(CKPT),map_location=device,weights_only=False)
ema_sd=ck.get('ema',ck['model']);model.load_state_dict(ema_sd);model.eval()

@torch.no_grad()
def ddim_sample(model,cond,ti_val,steps=25):
    B=cond.shape[0];ss,se=cond[:,:5],cond[:,5:]
    x_init=(1-ti_val)*ss+ti_val*se
    start=min(steps,T-1);noise=torch.randn_like(x_init)
    x=sac[start]*x_init+s1mac[start]*noise
    ti=torch.full((B,1),ti_val,device=device)
    stride=max(1,start//steps);ts=list(range(0,start,stride))[::-1]
    for i,tc in enumerate(ts):
        t=torch.full((B,),tc,device=device,dtype=torch.long)
        with autocast():np_=model(x,t,cond,ti)
        x0p=(x-s1mac[tc]*np_)/sac[tc].clamp(min=1e-5)
        if i<len(ts)-1:
            tn=ts[i+1];x=sac[tn]*x0p+s1mac[tn]*np_
        else:x=x0p
    return x

def label_rgb(s):
    r=np.zeros((*s.shape,3),dtype=np.float32)
    r[s==1]=[0.2,0.4,1.0];r[s==2]=[0.2,0.8,0.3];r[s==4]=[1.0,0.2,0.2];return r
def overlay(seg,t1c,a=0.6):
    tr=np.stack([t1c]*3,-1);sr=label_rgb(seg);m=sr.sum(-1,keepdims=True)>0
    return np.clip(np.where(m,(1-a)*tr+a*sr,tr),0,1)

# Visual test
fig,axes=plt.subplots(4,4,figsize=(16,16))
fig.suptitle('DDPMv3 DDIM Sampling on MRI',fontsize=14,fontweight='bold')
for j,c in enumerate(['Start','GT','Predicted','End']):axes[0,j].set_title(c,fontsize=12,fontweight='bold')
for row in range(4):
    s=va[row*5];cond=s['cond'].unsqueeze(0).to(device)
    ti=s['t_interp'].item();t1c=s['t1c'].numpy();gt=s['gt_labels'].numpy()
    pred=ddim_sample(model,cond,ti).argmax(1)[0].cpu().numpy()
    sl=s['cond'][:5].argmax(0).numpy();el=s['cond'][5:].argmax(0).numpy()
    d=dice_sc(torch.tensor(pred),torch.tensor(gt))
    for j,seg in enumerate([sl,gt,pred,el]):
        axes[row,j].imshow(overlay(seg,t1c),origin='lower');axes[row,j].axis('off')
    axes[row,2].set_xlabel(f'WT={d["WT"]:.2f} TC={d["TC"]:.2f} ET={d["ET"]:.2f}',fontsize=9)
plt.tight_layout();plt.savefig(str(OUT/'sampling_v3.png'),dpi=150,bbox_inches='tight');plt.show()

In [ ]:
# Interpolation sequence
s=va[0];cond=s['cond'].unsqueeze(0).to(device);t1c=s['t1c'].numpy()
fig,axes=plt.subplots(1,11,figsize=(22,2.5))
fig.suptitle('Tumor Evolution (DDPMv3 DDIM): t=0->1',fontsize=13,fontweight='bold')
for i,tv in enumerate(np.linspace(0,1,11)):
    pred=ddim_sample(model,cond,tv).argmax(1)[0].cpu().numpy()
    axes[i].imshow(overlay(pred,t1c),origin='lower')
    axes[i].set_title(f't={tv:.1f}',fontsize=9);axes[i].axis('off')
plt.tight_layout();plt.savefig(str(OUT/'interp_v3.png'),dpi=150,bbox_inches='tight');plt.show()

In [ ]:
# Anatomical Validation
def check_containment(seg):
    wt=seg>0;tc=(seg==1)|(seg==4);et=seg==4
    ev=(et&~tc).sum()/(et.sum()+1e-8)*100
    tv=(tc&~wt).sum()/(tc.sum()+1e-8)*100
    return {'et_viol%':ev,'tc_viol%':tv,'ok':ev<2 and tv<2}

def check_connectivity(seg):
    wt=(seg>0).astype(int)
    nc,_=scipy_label(wt)
    if nc==0:return{'n_comp':0,'clean':True}
    sizes=[((scipy_label(wt)[0])==i).sum() for i in range(1,nc+1)]
    return{'n_comp':nc,'largest%':max(sizes)/sum(sizes)*100,'clean':nc<=3}

def check_smoothness(seg):
    mask=(seg>0).astype(float)
    if mask.sum()==0:return{'compactness':0}
    perim=mask-binary_erosion(mask).astype(float)
    P=perim.sum();A=mask.sum()
    return{'compactness':min((4*np.pi*A)/(P**2+1e-8),1.0)}

# Run on 20 val samples
print('Anatomical Validation (20 val samples):')
print(f'{"Sample":>8} {"t":>5} {"WT":>6} {"TC":>6} {"ET":>6} {"Contain":>8} {"Comp#":>5} {"Compact":>8}')
for i in range(min(20,len(va))):
    s=va[i];cond=s['cond'].unsqueeze(0).to(device)
    ti=s['t_interp'].item();gt=s['gt_labels'].numpy()
    pred=ddim_sample(model,cond,ti).argmax(1)[0].cpu().numpy()
    d=dice_sc(torch.tensor(pred),torch.tensor(gt))
    ct=check_containment(pred);cn=check_connectivity(pred);sm=check_smoothness(pred)
    ok='✅' if ct['ok'] else '⚠'
    print(f'{i:8d} {ti:5.2f} {d["WT"]:6.3f} {d["TC"]:6.3f} {d["ET"]:6.3f} '
          f'{ok:>8} {cn["n_comp"]:5d} {sm["compactness"]:8.3f}')

In [ ]:
# Volume trajectory
s=va[0];cond=s['cond'].unsqueeze(0).to(device);t1c=s['t1c'].numpy()
vols={'WT':[],'TC':[],'ET':[],'t':[]}
for tv in np.linspace(0,1,21):
    pred=ddim_sample(model,cond,tv).argmax(1)[0].cpu().numpy()
    vols['WT'].append((pred>0).sum());vols['TC'].append(((pred==1)|(pred==4)).sum())
    vols['ET'].append((pred==4).sum());vols['t'].append(tv)
fig,ax=plt.subplots(figsize=(10,5))
for k,c in[('WT','#2E7D32'),('TC','#1565C0'),('ET','#E53935')]:
    ax.plot(vols['t'],vols[k],'-o',label=k,color=c,markersize=4)
ax.set_xlabel('Interpolation t');ax.set_ylabel('Volume (pixels)')
ax.set_title('Tumor Volume Trajectory',fontweight='bold')
ax.legend();ax.grid(True,alpha=0.3)
plt.tight_layout();plt.savefig(str(OUT/'volume_trajectory.png'),dpi=150);plt.show()

In [ ]:
# Summary
n_p=sum(p.numel() for p in model.parameters())/1e6
print('\n'+'='*60)
print('  DDPMv3 COMPLETE')
print('='*60)
print(f'  Model:    DDPMv3 ({n_p:.1f}M params, EMA, SelfAttn)')
print(f'  Best Dice:{best_dice:.4f}')
print(f'  Epochs:   {len(tl_)}')
print(f'  Files:')
for f in sorted(OUT.iterdir()):print(f'    {f.name:35s} {f.stat().st_size/1e6:.1f}MB')
print(f'  -> Next: Video_C_Validate.ipynb')
print('='*60)